# Step 3 - GNN Architecture and Training

This notebook trains four PyTorch models: GCN, GraphSAGE, GAT, and the homophily-gated GCN, from first-principles. It uses the sparse exports from `02-graph_engineering_baselines.ipynb`.

The experiment is transductive: all nodes and graph edges are used during message passing, but supervised loss is computed only on `labelled_train`. Validation macro-F1 selects the checkpoint; test scoring is intentionally separate.

In [1]:
from pathlib import Path
import sys
import torch
from IPython.display import display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from models import GNNClassifier
from pipeline_train import load_data, run_experiments

EXPORT_ROOT = ROOT / 'data/graph_engineering_50pc'
OUTPUT_ROOT = ROOT / 'data/gnn_training_50pc'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = [42, 123, 2024]
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)

PyTorch: 2.15.0
Device: cpu


## 1. Load the Step 2 contract

The loader checks the shared node order, split masks, graph, and budget-specific labels. Missing scarcity budgets are sampled per class from the original training pool with a fixed seed.

In [2]:
data = load_data(EXPORT_ROOT, budget='5pct', graph_name='knn', seed=42)
print('Features:', data.features.shape)
print('Edges:', data.edge_index.shape[1])
print('Labelled training nodes:', int(data.train_mask.sum()))
print('Validation nodes:', int(data.validation_mask.sum()))
print('Classes:', data.class_names)

Features: (65578, 50)
Edges: 1638524
Labelled training nodes: 1971
Validation nodes: 13116
Classes: ['CD14+ Monocyte', 'CD19+ B', 'CD34+', 'CD4+ T Helper2', 'CD4+/CD25 T Reg', 'CD4+/CD45RA+/CD25- Naive T', 'CD4+/CD45RO+ Memory', 'CD56+ NK', 'CD8+ Cytotoxic T', 'CD8+/CD45RA+ Naive Cytotoxic', 'Dendritic']


## 2. Smoke-test sparse layers

This small induced subgraph verifies that every architecture accepts weighted sparse edges and produces class logits without relying on PyTorch Geometric.

In [3]:
node_count = min(512, len(data.features))
edge_keep = (data.edge_index[0] < node_count) & (data.edge_index[1] < node_count)
small_x = torch.from_numpy(data.features[:node_count])
small_edges = torch.from_numpy(data.edge_index[:, edge_keep])
small_weights = torch.from_numpy(data.edge_weight[edge_keep])
for model_name in ['gcn', 'graphsage', 'gat', 'gated_gcn']:
    model = GNNClassifier(model_name, small_x.shape[1], 32, len(data.class_names), depth=2)
    logits = model(small_x, small_edges, small_weights)
    print(model_name, tuple(logits.shape))

gcn (512, 11)
graphsage (512, 11)
gat (512, 11)
gated_gcn (512, 11)


## 3. Main comparison

Set `RUN_MAIN = True` after the kernel and data exports are ready. The default uses both graph constructions, both existing label budgets, all four models, and three seeds. Results and checkpoints are written to `data/gnn_training_50pc/`.

In [ ]:
RUN_MAIN = True
if RUN_MAIN:
    results = run_experiments(
        EXPORT_ROOT, OUTPUT_ROOT,
        budgets=['5pct', 'full'],
        graph_names=['knn', 'snn'],
        model_names=['gcn', 'graphsage', 'gat', 'gated_gcn'],
        depths=[2],
        seeds=SEEDS,
        epochs=200,
        device=DEVICE,
    )
    display(results.sort_values(['budget', 'graph', 'macro_f1'], ascending=[True, True, False]))

## 4. Scarcity and depth ablations

The loader supports `1pct`, `2pct`, `5pct`, `10pct`, `20pct`, and `full`. Depth values are passed to the same training function so the comparison keeps graph, masks, optimizer, and checkpoint rules fixed. Run these as separate jobs if the full matrix is too large for one kernel session.

In [ ]:
RUN_ABLATIONS = True
if RUN_ABLATIONS:
    scarcity_results = run_experiments(
        EXPORT_ROOT, OUTPUT_ROOT / 'scarcity',
        budgets=['1pct', '2pct', '5pct', '10pct', '20pct', 'full'],
        graph_names=['knn'],
        model_names=['gcn', 'gated_gcn'],
        depths=[2],
        seeds=SEEDS,
        epochs=200,
        device=DEVICE,
    )
    depth_results = run_experiments(
        EXPORT_ROOT, OUTPUT_ROOT / 'depth',
        budgets=['5pct'],
        graph_names=['knn'],
        model_names=['gcn', 'gated_gcn'],
        depths=[1, 2, 3, 4],
        seeds=SEEDS,
        epochs=200,
        device=DEVICE,
    )
    display(scarcity_results)
    display(depth_results)

## 5. Reporting checklist

Report mean and standard deviation across seeds for accuracy and macro-F1, per-class F1 for all 11 classes, validation-selected epoch, and the gated model's mean gate value. Keep test scoring disabled until model, graph, depth, budget, and checkpoint rules are fixed.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Summarise the per-run validation results written above (mean and std over seeds).
frames = []
for experiment, folder in [("main", OUTPUT_ROOT), ("scarcity", OUTPUT_ROOT / "scarcity"), ("depth", OUTPUT_ROOT / "depth")]:
    path = folder / "validation_results.csv"
    if path.is_file():
        frames.append(pd.read_csv(path).assign(experiment=experiment))
runs = pd.concat(frames, ignore_index=True)
summary = (runs.groupby(["experiment", "budget", "graph", "model", "depth"], as_index=False)["macro_f1"]
               .agg(["mean", "std"]).rename(columns={"mean": "macro_f1_mean", "std": "macro_f1_std"}))

LABELS = {"gcn": "GCN", "graphsage": "GraphSAGE", "gat": "GAT", "gated_gcn": "Gated GCN"}
COLORS = {"gcn": "#2b6a99", "graphsage": "#3c9d7c", "gat": "#e08214", "gated_gcn": "#a23b72"}
PLOTS = OUTPUT_ROOT / "plots"
PLOTS.mkdir(exist_ok=True)

# 1. Main comparison: four models, two graphs, two budgets
main = summary[summary.experiment == "main"]
settings = [(b, g) for b in ["5pct", "full"] for g in ["knn", "snn"]]
fig, ax = plt.subplots(figsize=(12, 5))
bar_width = 0.2
for k, model in enumerate(LABELS):
    rows = [main[(main.budget == b) & (main.graph == g) & (main.model == model)].iloc[0] for b, g in settings]
    ax.bar(np.arange(len(settings)) + (k - 1.5) * bar_width, [r.macro_f1_mean for r in rows], bar_width,
           yerr=[r.macro_f1_std for r in rows], capsize=3, color=COLORS[model], label=LABELS[model])
ax.set_xticks(np.arange(len(settings)), [f"{b}\n{g.upper()}" for b, g in settings])
ax.set_ylabel("Validation macro-F1")
ax.set_title("Model comparison across graph and label-budget settings")
ax.legend(ncol=2)
fig.tight_layout(); fig.savefig(PLOTS / "model_comparison.png", dpi=150); plt.show()

# 2. Scarcity curve on the k-NN graph
scarcity = summary[summary.experiment == "scarcity"]
percent = {"1pct": 1, "2pct": 2, "5pct": 5, "10pct": 10, "20pct": 20, "full": 100}
fig, ax = plt.subplots(figsize=(8, 5))
for model in ["gcn", "gated_gcn"]:
    rows = scarcity[scarcity.model == model].assign(x=lambda d: d.budget.map(percent)).sort_values("x")
    ax.errorbar(rows.x, rows.macro_f1_mean, yerr=rows.macro_f1_std, marker="o", capsize=3,
                color=COLORS[model], label=LABELS[model])
ax.set_xscale("log")
ax.set_xticks(list(percent.values()), [str(v) for v in percent.values()])
ax.set_xlabel("Labeled training cells (%; full = 100)")
ax.set_ylabel("Validation macro-F1")
ax.set_title("Label scarcity on k-NN graph")
ax.legend()
fig.tight_layout(); fig.savefig(PLOTS / "scarcity_curve.png", dpi=150); plt.show()

# 3. Class-level gate versus local homophily (gated GCN, 5 % k-NN, seed 42)
stem = "gated_gcn_5pct_knn_depth2_seed42"
gate = np.load(OUTPUT_ROOT / f"{stem}_gate_values.npy").mean(axis=1)
gate_meta = pd.read_csv(OUTPUT_ROOT / f"{stem}_gate_values_metadata.csv").assign(gate=gate)
by_class = gate_meta.groupby("celltype")[["gate", "local_homophily"]].mean()
r = np.corrcoef(by_class.local_homophily, by_class.gate)[0, 1]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(by_class.local_homophily, by_class.gate, s=80, color=COLORS["gated_gcn"])
for name, row in by_class.iterrows():
    ax.annotate(name, (row.local_homophily, row.gate), xytext=(5, 3), textcoords="offset points", fontsize=9)
ax.set_xlabel("Mean local homophily")
ax.set_ylabel("Mean self-feature gate")
ax.set_title(f"Gate versus empirical homophily (r = {r:.2f})")
fig.tight_layout(); fig.savefig(PLOTS / "gate_vs_homophily.png", dpi=150); plt.show()

# 4. Training dynamics (5 % k-NN, depth 2, seed 42)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for model in LABELS:
    history = pd.DataFrame(json.loads((OUTPUT_ROOT / f"{model}_5pct_knn_depth2_seed42.json").read_text())["history"])
    axes[0].plot(history.epoch, history.loss, color=COLORS[model], label=LABELS[model])
    axes[1].plot(history.epoch, history.val_macro_f1, color=COLORS[model], label=LABELS[model])
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Cross-entropy loss")
axes[1].set(title="Validation macro-F1", xlabel="Epoch", ylabel="Macro-F1")
axes[1].legend()
fig.tight_layout(); fig.savefig(PLOTS / "training_dynamics.png", dpi=150); plt.show()
print("Plots saved to", PLOTS.relative_to(ROOT))
